In [ ]:
!pip install timm kaggle torch torchvision pillow opencv-python matplotlib


In [ ]:
import torch
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 6  # A, B-complex, C, D, E, K

model = timm.create_model(
    "efficientnet_b4",
    pretrained=False,
    num_classes=NUM_CLASSES
).to(device)


In [ ]:
from google.colab import files
files.upload()   # upload kaggle.json


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"shaikgousepeer","key":"fe3b161519d97fde8a190c3428daf679"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d udaykarthik21bce9252/vitamin-defficiency-dataset
!unzip vitamin-defficiency-dataset.zip -d raw_dataset


Streaming output truncated to the last 5000 lines.
  inflating: raw_dataset/dataset/Vitamin D deficiency/lentigo-adults-50.jpg  
  inflating: raw_dataset/dataset/Vitamin D deficiency/lentigo-adults-51.jpg  
  inflating: raw_dataset/dataset/Vitamin D deficiency/lentigo-adults-52.jpg  
  inflating: raw_dataset/dataset/Vitamin D deficiency/lentigo-adults-53.jpg  
  inflating: raw_dataset/dataset/Vitamin D deficiency/lentigo-adults-54.jpg  
  inflating: raw_dataset/dataset/Vitamin D deficiency/lentigo-adults-56.jpg  
  inflating: raw_dataset/dataset/Vitamin D deficiency/lentigo-adults-58.jpg  
  inflating: raw_dataset/dataset/Vitamin D deficiency/lentigo-adults-6.jpg  
  inflating: raw_dataset/dataset/Vitamin D deficiency/lentigo-adults-60.jpg  
  inflating: raw_dataset/dataset/Vitamin D deficiency/lentigo-adults-61.jpg  
  inflating: raw_dataset/dataset/Vitamin D deficiency/lentigo-adults-62.jpg  
  inflating: raw_dataset/dataset/Vitamin D deficiency/lentigo-adults-63.jpg  
  inflating: r

In [ ]:
FINAL_CLASSES = [
    "Vitamin_A",
    "Vitamin_B_Complex",
    "Vitamin_C",
    "Vitamin_D",
    "Vitamin_E",
    "Vitamin_K"
]


In [ ]:
import os, shutil

# ✅ Correct source path (IMPORTANT)
SRC = "raw_dataset/dataset"
DST = "clean_dataset"

os.makedirs(DST, exist_ok=True)

FINAL_CLASSES = [
    "Vitamin_A",
    "Vitamin_B_Complex",
    "Vitamin_C",
    "Vitamin_D",
    "Vitamin_E",
    "Vitamin_K"
]

def map_class(folder_name):
    name = folder_name.lower()
    if "vitamin a" in name:
        return "Vitamin_A"
    if "vitamin b" in name:
        return "Vitamin_B_Complex"
    if "vitamin c" in name:
        return "Vitamin_C"
    if "vitamin d" in name:
        return "Vitamin_D"
    if "vitamin e" in name:
        return "Vitamin_E"
    if "vitamin k" in name:
        return "Vitamin_K"
    return None  # noisy class (zinc/iron/etc)

counts = {c: 0 for c in FINAL_CLASSES}

for folder in os.listdir(SRC):
    src_folder = os.path.join(SRC, folder)
    if not os.path.isdir(src_folder):
        continue

    target_class = map_class(folder)
    if target_class is None:
        continue  # skip noisy folders

    dst_folder = os.path.join(DST, target_class)
    os.makedirs(dst_folder, exist_ok=True)

    for img in os.listdir(src_folder):
        if img.lower().endswith(('.jpg', '.jpeg', '.png')):
            shutil.copy(
                os.path.join(src_folder, img),
                os.path.join(dst_folder, f"{folder}_{img}")
            )
            counts[target_class] += 1

print("✅ CLEAN DATASET CREATED")
for k, v in counts.items():
    print(f"{k}: {v} images")


✅ CLEAN DATASET CREATED
Vitamin_A: 6499 images
Vitamin_B_Complex: 6987 images
Vitamin_C: 664 images
Vitamin_D: 9113 images
Vitamin_E: 692 images
Vitamin_K: 690 images


In [ ]:
import os

print("\nVerification:\n")
for cls in os.listdir("clean_dataset"):
    cls_path = os.path.join("clean_dataset", cls)
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith(('.jpg','.jpeg','.png'))]
    print(cls, "→", len(imgs), "images")



Verification:

Vitamin_B_Complex → 6987 images
Vitamin_D → 9113 images
Vitamin_K → 690 images
Vitamin_C → 664 images
Vitamin_A → 6499 images
Vitamin_E → 692 images


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

transform = transforms.Compose([
    transforms.Resize((380,380)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

dataset = datasets.ImageFolder(DST, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False)

CLASS_NAMES = dataset.classes


In [ ]:
import torch, timm, torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = timm.create_model(
    "efficientnet_b4",
    pretrained=True,
    num_classes=len(CLASS_NAMES)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PHASE1_PATH = "/content/drive/MyDrive/vitamin_phase1_efficientnet_b4.pth"

model.load_state_dict(torch.load(PHASE1_PATH, map_location=device))
model.eval()

print("✅ Phase-1 model loaded successfully")


✅ Phase-1 model loaded successfully


In [ ]:
for p in model.parameters():
    p.requires_grad = False


In [ ]:
# Unfreeze last 2 blocks
for p in model.blocks[-2:].parameters():
    p.requires_grad = True

# Unfreeze classifier
for p in model.classifier.parameters():
    p.requires_grad = True


In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)


In [ ]:
criterion = torch.nn.CrossEntropyLoss()


In [ ]:
import torch

def train_epoch(loader):
    model.train()
    correct, total = 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)

    return correct / total


def eval_epoch(loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)

            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)

    return correct / total


In [ ]:
print(model is not None)
print(train_loader is not None)
print(val_loader is not None)
print(optimizer is not None)
print(criterion is not None)


True
True
True
True
True


In [ ]:
EPOCHS_PHASE2 = 8  # 6–10 is ideal

for e in range(EPOCHS_PHASE2):
    train_acc = train_epoch(train_loader)
    val_acc = eval_epoch(val_loader)

    print(f"[Phase-2] Epoch {e+1}: Train={train_acc:.4f}, Val={val_acc:.4f}")


[Phase-2] Epoch 1: Train=0.8138, Val=0.8194
[Phase-2] Epoch 2: Train=0.8259, Val=0.8172
[Phase-2] Epoch 3: Train=0.8323, Val=0.8087
[Phase-2] Epoch 4: Train=0.8438, Val=0.8048
[Phase-2] Epoch 5: Train=0.8447, Val=0.7987
[Phase-2] Epoch 6: Train=0.8495, Val=0.7900
[Phase-2] Epoch 7: Train=0.8511, Val=0.7854
[Phase-2] Epoch 8: Train=0.8569, Val=0.7778


In [ ]:
FINAL_PATH = "/content/drive/MyDrive/vitamin_deficiency_efficientnet_b4_final.pth"

torch.save(model.state_dict(), FINAL_PATH)

print("✅ FINAL model saved (Phase-1 + Phase-2 combined)")


✅ FINAL model saved (Phase-1 + Phase-2 combined)
